In [ ]:

import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import sys
sys.path.append("multiHIVE/src")
from multiHIVE import multiHIVE
import torch

In [9]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
torch.set_float32_matmul_precision("high")

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
adata = sc.read_h5ad( "/Data/TEA-seq/TEA-seq.h5ad")
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 25517 × 165454
    obs: 'cell_type', 'batch'
    var: 'modality'
    obsm: 'protein_expression'

In [11]:
del adata.obsm
del adata.obsp

In [ ]:
adata = scvi.data.organize_multiome_anndatas(adata)
adata = adata[:, adata.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata, min_cells=int(adata.shape[0] * 0.01))
multiHIVE.setup_anndata(adata, batch_key="modality")
adata

In [ ]:
vae = multiHIVE(adata, latent_distribution="normal",
                n_genes=(adata.var["modality"] == "Gene Expression").sum(),
                n_regions=(adata.var["modality"] == "Peaks").sum(),
                n_proteins=0,
               )

In [ ]:
vae.train()

In [15]:
vae.get_latent_representation()

In [18]:
adata.write("./outputs/tea-seq/multiHIVE.h5ad")